## 基础知识

#### State Update

##### 更新原则

- 对于节点没有返回的字段，保留原有状态值
- 对于节点返回的字段，看是否配置了 reducer
   - 配置了，使用 reducer 合并，如果某个节点使用了 Overwrite，则不使用 reducer
   - 没配置，按默认覆盖处理

In [ ]:
from operator import add
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langgraph.types import Overwrite


class State(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: State) -> State:
    print(state['cur_id'])
    return {
        'logs': Overwrite(['node_1执行完成']),
        'cur_id': 'node_1'
    }

def node_2(state: State) -> State:
    print(state['cur_id'])
    return {
        'logs': ['node_2执行完成'],
        'cur_id': 'node_2'
    }

builder = StateGraph(State)
builder.add_node(node_1)
builder.add_node(node_2)
builder.add_edge(START, 'node_1')
builder.add_edge('node_1', 'node_2')
builder.add_edge('node_2', END)
graph = builder.compile()
result = graph.invoke({'logs': ['START执行完成'], 'cur_id': 'start'})
print(result)

#### 并行节点合并更新

返回的字段必须配置 reducer 函数